# 01_Data_Preparation.ipynb - Framingham+ Data Preparation for Cardiovascular Risk Prediction

## Objective
This notebook loads, cleans, and prepares the Framingham Heart Study dataset for machine learning. The output will be used in the next notebook (`02_Model_Training.ipynb`) to train an XGBoost model for 10-year cardiovascular risk prediction.

## Dataset
- **Source:** Framingham Heart Study
- **Patients:** 4,240
- **Variables:** 16 (demographics, lifestyle, medical history, clinical measurements)
- **Target:** `TenYearCHD` (0 = no CHD event within 10 years, 1 = CHD event)

## Workflow in this notebook
1. Load the dataset
2. Explore basic statistics and missing values
3. Handle missing values (median imputation for numeric columns)
4. Split data into training (80%) and test (20%) sets (stratified by target)
5. Save processed data for the training notebook


## Expected outputs
- `X_train`, `X_test`, `y_train`, `y_test` saved as a pickle file
- Dataset shape and target distribution summary

In [ ]:
#import drive
from google.colab import drive
#Import the relevant libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
## Saving and loading large models or objects efficiently
import joblib

In [ ]:
# 1. Load dataset
df = pd.read_csv('/content/drive/MyDrive/framingham_plus_notebooks/heart_disease.csv')

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset loaded successfully
Shape: (4238, 16)
Columns: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']


In [ ]:
# 2. Basic exploration
print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget distribution (TenYearCHD):")
print(df['TenYearCHD'].value_counts())
print(f"CHD prevalence: {df['TenYearCHD'].mean():.3f}")


First 5 rows:
   male  age  education  currentSmoker  cigsPerDay  BPMeds  prevalentStroke  \
0     1   39        4.0              0         0.0     0.0                0   
1     0   46        2.0              0         0.0     0.0                0   
2     1   48        1.0              1        20.0     0.0                0   
3     0   61        3.0              1        30.0     0.0                0   
4     0   46        3.0              1        23.0     0.0                0   

   prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  heartRate  glucose  \
0             0         0    195.0  106.0   70.0  26.97       80.0     77.0   
1             0         0    250.0  121.0   81.0  28.73       95.0     76.0   
2             0         0    245.0  127.5   80.0  25.34       75.0     70.0   
3             1         0    225.0  150.0   95.0  28.58       65.0    103.0   
4             0         0    285.0  130.0   84.0  23.10       85.0     85.0   

   TenYearCHD  
0           0  
1  

Due to the small size of the dataset, NaN values will not be dropped. Instead, they will be handled using the following strategy:

For numerical values, missing data will be imputed using the median, as it is more robust to outliers than the mean.

For binary values, missing data will be imputed using the mode, since this preserves the most frequent category and avoids introducing artificial values.

For categorical variables, missing data will also be imputed using the mode.

No rows will be removed, in order to preserve as much information as possible for model training.

In [ ]:
# 3. Handle missing values
from sklearn.impute import SimpleImputer

binary_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']
categorical_cols = ['education']
feature_cols = ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay',
                'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes',
                'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
numeric_cols = [col for col in feature_cols if col not in categorical_cols + binary_cols]

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
bin_imputer = SimpleImputer(strategy='most_frequent')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])
df[binary_cols] = bin_imputer.fit_transform(df[binary_cols])

print('\nMissing values after imputation:')
print(df.isnull().sum().sum())


Missing values after imputation:
0


In [ ]:
# 4. Define features and target
X = df[feature_cols]
y = df['TenYearCHD']

print(f'\nFeatures shape: {X.shape}')
print(f'Target shape: {y.shape}')



Features shape: (4238, 15)
Target shape: (4238,)


Now that features and target values have been defined and separated, it's time to split the data into two sets: one for model training and the other for testing.
 Stratifying the split preserves the original class distribution in both sets, avoids imbalance‑related bias, and ensures a more reliable evaluation of model performance.



In [ ]:
# 5. Train-test split (stratified to preserve target distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')
print(f'Training CHD prevalence: {y_train.mean():.3f}')
print(f'Test CHD prevalence: {y_test.mean():.3f}')


Training set size: 3390
Test set size: 848
Training CHD prevalence: 0.152
Test CHD prevalence: 0.152


In [ ]:

# 6. Save processed data
processed_data = (X_train, X_test, y_train, y_test)
joblib.dump(processed_data, '/content/drive/MyDrive/framingham_plus_notebooks/framingham_processed.pkl')
print("\n✅ Data preparation complete. Processed data saved to 'framingham_processed.pkl'")



✅ Data preparation complete. Processed data saved to 'framingham_processed.pkl'
